# Processing buffer with output assertions

Run a native QGIS Processing algorithm against synthetic data, then validate the result rather than relying on a success message. Requires Processing initialized in the QGIS kernel.


## Learning objective
Use a small, deterministic example to verify spatial behavior and make assumptions explicit.


In [ ]:
import processing
from qgis.core import QgsFeature, QgsGeometry, QgsPointXY, QgsVectorLayer

points = QgsVectorLayer('Point?crs=EPSG:3857', 'sample_points', 'memory')
feature = QgsFeature(points.fields())
feature.setGeometry(QgsGeometry.fromPointXY(QgsPointXY(0, 0)))
points.dataProvider().addFeatures([feature])
points.updateExtents()
buffered = processing.run('native:buffer', {
    'INPUT': points, 'DISTANCE': 25, 'SEGMENTS': 8,
    'END_CAP_STYLE': 0, 'JOIN_STYLE': 0, 'MITER_LIMIT': 2,
    'DISSOLVE': False, 'OUTPUT': 'memory:',
})['OUTPUT']
assert buffered.isValid() and buffered.featureCount() == 1
result_geometry = next(buffered.getFeatures()).geometry()
assert not result_geometry.isEmpty() and result_geometry.area() > 0
print('Buffer output validated; area in projected CRS units squared:', result_geometry.area())

# This is a synthetic demonstration; inspect CRS units before using real distances.